<center><h1>Timmala_Karan_HW3</h1></center>
<br>
<br>

Name: Karan Timmala
<br>
Github Username: karantimmala
<br>
USC ID: 1373267232

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Download the AReM data from: https://archive.ics.uci.edu/ml/datasets/
Activity+Recognition+system+based+on+Multisensor+data+fusion+\%28AReM\
%29 . The dataset contains 7 folders that represent seven types of activities. In
each folder, there are multiple files each of which represents an instant of a human
performing an activity.1 Each file containis 6 time series collected from activities
of the same person, which are called avg rss12, var rss12, avg rss13, var rss13,
vg rss23, and ar rss23. There are 88 instances in the dataset, each of which contains 6 time series and each time series has 480 consecutive values.

Package imports

Get the AReM Data Set

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import resample

### (b) Test and Train Data

Keep datasets 1 and 2 in folders bending1 and bending 2, as well as datasets 1,
2, and 3 in other folders as test data and other datasets as train data.

In [2]:
# Root directory where AReM data is stored
root = "../data/AReM"

# Column names for each dataset
col_names = [
    "time",
    "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23"
]

# Initialize empty lists to store the pandas dataFrames assigned to the training and test sets
train_data = []
test_data = []
train_labels = []
test_labels = []

# Iterate through every item in the root folder.
for ite in sorted(os.listdir(root)):
    
    ite_path = os.path.join(root, ite)
    
    # Skip non-folder files
    if not os.path.isdir(ite_path):
        continue
    
    # Iterate through each file present within the current activity folder
    for file in os.listdir(ite_path):
        
        # Check if the file ends with '.csv', skip any other file format
        if not file.endswith(".csv"):
            continue
        
        file_loc = os.path.join(ite_path, file)
        
        # Read CSV while skipping metadata rows
        df = pd.read_csv(
            file_loc,skiprows=5,on_bad_lines="skip",header=None,index_col=None,
            names=col_names,sep=None,skipinitialspace=True,engine="python"
        ).iloc[:, :7]
        
        # Extract dataset number from filename
        dataset_num = int(
            file.replace("dataset", "").replace(".csv", "")
        )
        
        # Check if the current activity folder is 'bending1' or 'bending2'
        if ite in ["bending1", "bending2"]:
            
            # For bending activities, assign datasets 1 and 2 to the testing set
            if dataset_num in [1, 2]:
                test_data.append(df)
                test_labels.append(ite)
            else:
                train_data.append(df)
                train_labels.append(ite)
        
        else:
            
            # For all other activity categories, assign datasets 1, 2, and 3 to testing
            if dataset_num in [1, 2, 3]:
                test_data.append(df)
                test_labels.append(ite)
            else:
                train_data.append(df)
                train_labels.append(ite)

# Print total number of datasets in train and test datasets
print(f"Datasets in Test Data: ", len(test_data))
print(f"Datasets in Train Data: ", len(train_data))

Datasets in Test Data:  19
Datasets in Train Data:  69


In [3]:
def check_missing(data, label="Instance"):

    # Iterate over the list of DataFrames with 1-based indexing for display numbers
    for ite, df in enumerate(data, start=1):

        # Calculate total missing values in the current DataFrame:
        # df.isna() creates a DataFrame of booleans (True for NaN/missing values)
        
        nan_count = df.isna().sum().sum()

        # Check if the total count of missing values in the DataFrame is greater than 0
        if nan_count > 0:
            print(f"{label} {ite}: {nan_count} NaN values")

## Checks and prints which instance contains missing values
check_missing(test_data, "Missing values in Test instance")
check_missing(train_data, "Missing values in Train instance")

Missing values in Train instance 7: 480 NaN values


In [4]:
train_miss_inst = train_data[6]  
train_miss_inst

,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,32.50,0.50,0.00,0.00,19.00,1.00,NaN
250,32.50,0.50,0.00,0.00,18.50,0.50,NaN
500,32.75,0.43,1.00,0.00,18.00,0.00,NaN
750,32.50,0.50,0.00,0.00,17.50,0.50,NaN
1000,32.50,0.50,7.50,0.50,17.50,0.87,NaN
...,...,...,...,...,...,...,...
118750,28.67,0.47,4.67,1.25,17.33,0.47,NaN
119000,27.50,0.50,5.50,2.50,17.25,1.30,NaN
119250,28.00,0.00,6.67,0.94,17.00,1.00,NaN
119500,28.00,0.00,5.00,0.82,17.00,0.71,NaN


### (c) Feature Extraction

#### i. Research what types of time-domain features are usually used in time series classification and list them (examples are minimum, maximum, mean, etc).


Min & Max: Mark the lower and upper boundaries of the signal amplitude.

Mean & Median: Represent central trend metrics to establish the signal's baseline level.

Standard Deviation & Variance: Quantify variability and fluctuation in signal energy or amplitude.

Percentiles / Interquartile Range (IQR): Measure data spread at given thresholds (e.g., 25th and 75th percentiles), offering outlier-resistant dispersion metrics.

Skewness: Gauges the degree of asymmetry in the data distribution around the mean.

Kurtosis: Indicates the presence of heavy tails or extreme outlier values in the signal.

Mean Absolute Deviation (MAD): Average distance of points from the mean.

Root Mean Square (RMS): Quantifies the overall power/magnitude of the signal.

Entropy: Quantifies the degree of complexity, regularity, or randomness in the sequence.



#### ii. Extract the time-domain features minimum, maximum, mean, median, standard deviation, first quartile, and third quartile for all of the 6 time series in each instance. You are free to normalize/standardize features or use them directly.


In [5]:
# Combine the list of training DataFrames with the list of testing DataFrames into a single list
comb_data = train_data + test_data
comb_labels = train_labels + test_labels

# Initialize an empty list to store the extracted feature vector for each dataset instance
feature_mat = []

# Loop through each DataFrame instance in the combined dataset list
for df in comb_data:
    # Drop time column
    sig_data = df.drop(columns=["time"])
    
    # Compute summary statistics (min, max, mean, median, std) for each remaining column
    statistic_frame = sig_data.agg(["min", "max", "mean", "median", "std"])
    
    # Calculate the 25th percentile (1st quartile) and 75th percentile (3rd quartile) for each column
    statistic_frame.loc["q1"] = sig_data.quantile(0.25)
    statistic_frame.loc["q3"] = sig_data.quantile(0.75)
    
    # Reorder the rows of summary statistics into the required specific order
    statistic_frame = statistic_frame.loc[
        ["min", "max", "mean", "median", "std", "q1", "q3"]
    ]
    
    # Transpose the DataFrame, flatten the 2D matrix into a 1D array, and append it to feature_matrix
    feature_mat.append(statistic_frame.T.values.flatten())


# Generate Column Names Based on Signal Names
sig_names = train_data[0].columns[1:]  

feature_col = []

# Iterate over the 6 time series signals to generate feature column names
for ite in range(1, 7):
    feature_col.extend([
        f"{ite}-MIN",
        f"{ite}-MAX",
        f"{ite}-MEAN",
        f"{ite}-MEDIAN",
        f"{ite}-STD",
        f"{ite}-Q1",
        f"{ite}-Q3"
    ])

# Create DataFrame
feature_df = pd.DataFrame(feature_mat, columns=feature_col)

# Reindex the DataFrame to use 1-based indexing (1 to N) instead of 0-based
feature_df.index = range(1, len(feature_df) + 1)
feature_df.index.name = "Instance"

display(feature_df)

,1-MIN,1-MAX,1-MEAN,1-MEDIAN,1-STD,1-Q1,1-Q3,2-MIN,2-MAX,2-MEAN,...,5-STD,5-Q1,5-Q3,6-MIN,6-MAX,6-MEAN,6-MEDIAN,6-STD,6-Q1,6-Q3
Instance,,,,,,,,,,,,,,,,,,,,,
1,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.500,0.0,1.79,0.493292,0.430,0.513506,0.000,0.940
2,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.330,0.0,2.18,0.613521,0.500,0.524317,0.000,1.000
3,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.250,0.0,1.79,0.383292,0.430,0.389164,0.000,0.500
4,37.00,48.00,43.454958,43.250,1.386098,42.50,45.0000,0.0,1.58,0.378083,...,2.488862,22.2500,24.000,0.0,5.26,0.679646,0.500,0.622534,0.430,0.870
5,36.25,48.00,43.969125,44.500,1.618364,43.31,44.6700,0.0,1.50,0.413125,...,3.318301,20.5000,23.750,0.0,2.96,0.555312,0.490,0.487826,0.000,0.830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,35.50,46.25,43.174938,43.670,1.989052,42.50,44.5000,0.0,2.12,0.506583,...,2.983976,12.7500,16.500,0.0,5.72,0.911979,0.830,0.666161,0.470,1.220
85,32.75,47.00,42.760562,44.500,3.398919,41.33,45.3725,0.0,3.34,0.486167,...,4.296574,13.0000,18.565,0.0,5.73,0.842271,0.710,0.722165,0.430,1.090
86,19.33,43.50,34.227771,35.500,4.889576,30.50,37.7500,0.0,14.50,3.995729,...,3.092094,14.7500,18.670,0.0,9.74,3.394125,3.100,1.792090,2.105,4.425


#### iii. Standard Deviation Estimate the standard deviation of each of the time-domain features you extracted from the data. Then, use Python’s bootstrapped or any other method to build a 90% bootsrap confidence interval for the standard deviation of each feature.

In [6]:
feature_df.std(numeric_only=True)

1-MIN       10.057319
1-MAX        6.334222
1-MEAN       6.678082
1-MEDIAN     6.777133
1-STD        1.785060
1-Q1         7.264518
1-Q3         6.632170
2-MIN        0.000000
2-MAX        5.104948
2-MEAN       1.723475
2-MEDIAN     1.614151
2-STD        0.917441
2-Q1         1.070117
2-Q3         2.337646
3-MIN        2.956462
3-MAX        5.180639
3-MEAN       4.218631
3-MEDIAN     4.264695
3-STD        0.996873
3-Q1         4.358157
3-Q3         4.471419
4-MIN        1.172604
4-MAX        2.790182
4-MEAN       2.117172
4-MEDIAN     2.113140
4-STD        0.461634
4-Q1         1.927301
4-Q3         2.381122
5-MIN        6.105679
5-MAX        6.050019
5-MEAN       5.901249
5-MEDIAN     6.041312
5-STD        1.058649
5-Q1         6.250285
5-Q3         5.840416
6-MIN        0.046101
6-MAX        2.533515
6-MEAN       1.157299
6-MEDIAN     1.089858
6-STD        0.517002
6-Q1         0.761645
6-Q3         1.527012
dtype: float64

In [7]:
# Ensure dataframe is numeric
features_num = feature_df.copy()
features_num = features_num.apply(pd.to_numeric, errors="coerce")   # # Convert all values to numbers, turning non-convertible items into NaN

B_num = 1000          # Set the number of bootstrap resampling iterations
low = 5     # Define the lower percentile threshold (5th percentile) for a 90% confidence interval
high = 95   # Define the upper percentile threshold (95th percentile) for a 90% confidence interval

res = []      # Initialize an empty list to store output statistics for each feature

np.random.seed(42)   # Fix the random number generator seed so results are reproducible

# # Iterate through each column in the numerical DataFrame and Compute Std and Bootstrap CI for Each Feature
for ite in features_num.columns:
    
    # Remove NaN values for current feature
    clean_val = features_num[ite].dropna().values    # Filter out missing values and extract the remaining numbers as a NumPy array
    x = len(clean_val)
    
    # If less than 2 values, cannot compute std
    if x < 2:      # Check if there are fewer than 2 data points (insufficient to calculate sample std)
        res.append([ite, np.nan, np.nan, np.nan])    # Store NaN for standard deviation and confidence interval bounds
        continue
    
    # Compute original sample standard deviation
    sample_stand = np.std(clean_val, ddof=1)     # Calculate the sample standard deviation using sample variance correction (ddof=1)
    
    # Generate bootstrap distribution of std
    boot_stds = []
    
    for _ in range(B_num):
        random_ind = np.random.randint(0, x, x)
        boot_sample = clean_val[random_ind]   # Select the data values corresponding to the generated random indices
        boot_stds.append(np.std(boot_sample, ddof=1))  # Compute sample std for this resampled dataset and append to list
    
    boot_stds = np.array(boot_stds)
    
    # Calculate the 5th percentile as the lower confidence bound and 95th percentile as Upper confidence bond
    conf_lower = np.percentile(boot_stds, low)
    conf_upper = np.percentile(boot_stds, high)
    
    res.append([ite, sample_stand, conf_lower, conf_upper])

# Convert the accumulated list of metrics into a new pandas DataFrame
boot_res = pd.DataFrame(
    res,
    columns=[
        "FEATURE",
        "SAMPLE STD",
        "CONF_INTER LOWER(5%)",
        "CONF_INTER UPPER(95%)"
    ]
)

boot_res

,FEATURE,SAMPLE STD,CONF_INTER LOWER(5%),CONF_INTER UPPER(95%)
0,1-MIN,10.057319,8.606643,11.356208
1,1-MAX,6.334222,3.469555,9.120377
2,1-MEAN,6.678082,4.808602,8.752119
3,1-MEDIAN,6.777133,4.870043,8.907726
4,1-STD,1.785060,1.580643,1.963604
5,1-Q1,7.264518,5.716312,9.059727
6,1-Q3,6.632170,4.410619,8.862805
7,2-MIN,0.000000,0.000000,0.000000
8,2-MAX,5.104948,4.677218,5.422608
9,2-MEAN,1.723475,1.463301,1.974155


#### iv. Use your judgement to select the three most important time-domain features (one option may be min, mean, and max).

##### Based on my judgement, I selected Mean, Maximum and Standard Deviation as the three most important time-domain features.
##### **Maximum:** The maximum value records the highest intensity peak within a given time frame. Even when different activities share similar average values, their peak outputs often differ sharply, allowing this feature to highlight sudden, high-intensity movements.
##### **Mean:** The mean establishes the signal's core baseline over time. Because distinct physical activities typically function at varying baseline intensities, this metric makes it easy to separate actions with sustained high intensity from those with lower overall activity levels.
##### **Standard Deviation:** Standard deviation measures the degree of variation and instability within the signal. Continuous physical movement generally causes greater signal fluctuation, while stationary positions keep the signal stable, making variability a key metric for distinguishing between active and idle states.

##### Together, these three parameters capture central tendency, variance, and peak values, providing a concise yet effective feature set to differentiate time-series signal behaviors. 

# 2. ISLR 3.7.4

### I collect a set of data (n = 100 observations) containing a single predictor and a quantitative response. I then fit a linear regression model to the data, as well as a separate cubic regression, i.e. Y = β0 + β1X + β2X2 + β3X3 + ϵ.


### (a) Suppose that the true relationship between X and Y is linear, i.e. Y = β0 + β1X + ϵ. Consider the training residual sum of squares (RSS) for the linear regression, and also the training RSS for the cubic regression. Would we expect one to be lower than the other, would we expect them to be the same, or is there not enough information to tell? Justify your answer.

##### Ans : When the true underlying relationship between $X$ and $Y$ is linear, a linear regression model accurately captures the underlying process. Nevertheless, a cubic regression model inherently encompasses the linear model when its higher-order coefficients ($\beta_2$ and $\beta_3$) equal zero. Because of these extra parameters, the cubic model possesses greater capacity to fit the noise in the training set. Consequently, its training residual sum of squares (RSS) will always be lower than or equal to that of the linear model, as the increased flexibility allows it to fit the training data more tightly even if the actual pattern is linear.

### **(b) Answer (a) using test rather than training RSS.**


##### Ans : When evaluating test RSS, the outcome shifts in favor of the simpler model. Because the actual data-generating process is linear, the linear regression achieves minimal bias without introducing redundant complexity. Conversely, the cubic regression introduces extra degrees of freedom, which inflates the model's variance and leaves it prone to overfitting by fitting random noise present in the training set. Consequently, the linear model is anticipated to achieve a lower test RSS compared to the cubic model.

### 
**(c) Suppose that the true relationship between X and Y is not linear, but we don’t know how far it is from linear. Consider the training RSS for the linear regression, and also the training RSS for the cubic regression. Would we expect one to be lower than the other, would we expect them to be the same, or is there not enough information to tell? Justify your answer.**

##### Ans : When the actual relationship between $X$ and $Y$ is non-linear, the cubic regression provides greater flexibility than the linear model. Since the cubic specification encompasses the linear model as a subset where $\beta_2 = \beta_3 = 0$, it can adapt to the training observations at least as effectively as the simpler model. Consequently, the cubic regression will yield a training RSS that is less than or equal to that of the linear model.

### **(d)Answer (c) using test rather than training RSS.**

##### Ans : When evaluating test RSS under non-linear conditions, there is insufficient information to conclusively determine which model will outperform the other. If the true data-generating process is only slightly non-linear, the linear regression may yield better generalization due to its minimal model variance. Conversely, if the non-linear structure is pronounced, the cubic regression's ability to significantly lower estimation bias could outweigh its higher variance, resulting in a superior test RSS. Consequently, relative performance depends on the underlying bias–variance tradeoff.

### AI USED

1.How can a 90% percentile-based bootstrap confidence interval for a feature's standard deviation be implemented programmatically in Python?

2.What are the best practices for managing missing values ($NaNs$) when calculating sample standard deviations and bootstrap estimates?

3.What is the standard Python approach for calculating summary statistics (min, max, mean, median, standard deviation, and the 25th and 75th percentiles) across sensor channels?

4.What are the standard time-domain statistical indicators used when feature-engineering time-series classification models?

### REFERENCES

https://nerdnodes2023.medium.com/feature-engineering-for-time-series-classification-a-comprehensive-guide-f2073084718a

https://machinelearningmastery.com/feature-selection-time-series-forecasting-python/

https://machinelearningmastery.com/how-to-calculate-the-5-number-summary-for-your-data-in-python/

https://statisticsfundamentals.com/sampling-distributions/bootstrap-sampling/

https://www.dasca.org/world-of-data-science/article/strategies-for-handling-missing-values-in-data-analysis